# 例 7.3 · KO 与 PEP 的协整和相关性

可口可乐（KO）和百事（PEP）的股价是否存在长期均衡关系？它们的日收益率又有多大程度的同向变动？本例分别用价格和收益率回答这两个问题。[MATLAB 原例](../matlab/example7_3.m) · [Python 实现](../python/examples/example7_3.py)

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.cadf import cadf
from epchan.get_tday import get_tday
from epchan.ols import ols
from epchan.paths import data_path
from epchan.prt import prt

## 1. 对齐复权收盘价

取两份价格表的共同交易日，按时间升序排列，只保留两只股票都有有限价格的日期。后续检验和作图都使用这个样本。

In [ ]:
ko_dates, ko_prices = get_tday(data_path('KO.xls'))
pep_dates, pep_prices = get_tday(data_path('PEP.xls'))
dates, ko_idx, pep_idx = np.intersect1d(ko_dates, pep_dates, return_indices=True)
ko, pep = ko_prices[ko_idx], pep_prices[pep_idx]
valid = np.isfinite(ko) & np.isfinite(pep)
dates, ko, pep = dates[valid], ko[valid], pep[valid]
print(f'共同有效交易日共 {len(dates)} 天，从 {dates[0]} 到 {dates[-1]}。')

## 2. 检验价格序列是否协整

以 KO 为因变量、PEP 为解释变量，协整回归含常数项（`p=0`）。对残差做增广 Dickey–Fuller 检验，加入一阶滞后差分（`l=1`）。统计量低于某个显著性水平的临界值时，才能在该水平拒绝无协整原假设。

In [ ]:
result = cadf(ko, pep, p=0, l=1)
prt(result, ['KO', 'PEP'])

## 3. 估计对冲比率并查看价差

原例另做一次无截距回归 $KO_t = \beta PEP_t + z_t$。每持有 1 股 KO、卖空 $\beta$ 股 PEP，按复权价格计算的组合价差就是 $z_t$。下图对应书中图 7.2。

In [ ]:
fit = ols(ko, pep)
hedge_ratio = float(fit.beta[0, 0])
spread = fit.resid.ravel()
print(f'对冲比率 = {hedge_ratio:.4f} 股 PEP / 股 KO')

In [ ]:
trading_days = [datetime.strptime(str(day), '%Y%m%d').date() for day in dates]
fig, ax = plt.subplots(figsize=(9, 3.6), layout='constrained')
ax.axhline(0, color='0.65', linewidth=0.8, linestyle='--')
ax.plot(trading_days, spread, color='#286BB0', linewidth=1.5)
ax.set(
    title=f'KO minus {hedge_ratio:.4f} shares of PEP',
    xlabel='Trading date', ylabel='Spread (USD per KO share)',
)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

## 4. 检验日收益率的相关性

用相邻两天的复权价格计算简单收益率。相关矩阵的非对角元素给出 KO 与 PEP 日收益率的 Pearson 相关系数。双侧检验的原假设是总体相关系数为零。

In [ ]:
prices = np.column_stack((ko, pep))
daily_returns = np.diff(prices, axis=0) / prices[:-1]
correlation = np.corrcoef(daily_returns, rowvar=False)
p_value = pearsonr(daily_returns[:, 0], daily_returns[:, 1]).pvalue
print(f'日收益率相关矩阵（KO, PEP）\n{correlation.round(4)}')
print(f'双侧检验 p < 0.001：{p_value < 0.001}')

### 结果解读

KO 与 PEP 的日收益率呈正相关（**0.4849**）。价格协整检验的统计量为 **−2.1426**，高于 10% 临界值 **−3.038**。这个样本中，两只股票的日涨跌存在关联，但缺乏支持长期价格均衡的统计证据。